In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load CSV files into DataFrames (keep full DataFrames in memory for later steps)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

# Helper timezone and functions
tz = pytz.timezone('Asia/Shanghai')

def summarize_df(df, ts_col='timestamp', name_col=None, name_label='name', top_cmdb=20, top_names=50):
    # Ensure timestamp is integer
    df[ts_col] = pd.to_numeric(df[ts_col], errors='coerce').astype('Int64')
    # Compute row count and min/max epoch
    rows = int(df.shape[0])
    if rows == 0:
        min_epoch = None
        max_epoch = None
        min_local = None
        max_local = None
    else:
        min_epoch = int(df[ts_col].min())
        max_epoch = int(df[ts_col].max())
        # Convert to timezone-aware datetimes (assume epoch UTC then convert to Asia/Shanghai)
        min_local = pd.to_datetime(min_epoch, unit='s', utc=True).tz_convert(tz)
        max_local = pd.to_datetime(max_epoch, unit='s', utc=True).tz_convert(tz)
    # Top cmdb_id counts (alphabetical when counts equal)
    if 'cmdb_id' in df.columns:
        top_cmdb = df['cmdb_id'].value_counts().sort_index().sort_values(ascending=False).head(top_cmdb)
        top_cmdb_df = top_cmdb.reset_index()
        top_cmdb_df.columns = ['cmdb_id', 'count']
    else:
        top_cmdb_df = pd.DataFrame(columns=['cmdb_id','count'])
    # Name/counts summary
    if name_col and name_col in df.columns:
        name_counts = df[name_col].value_counts().head(top_names).reset_index()
        name_counts.columns = [name_label, 'count']
    else:
        name_counts = pd.DataFrame(columns=[name_label, 'count'])
    # Build compact summary DataFrame
    summary = pd.DataFrame([{
        'rows': rows,
        'min_epoch': min_epoch,
        'max_epoch': max_epoch,
        'min_local (Asia/Shanghai)': min_local,
        'max_local (Asia/Shanghai)': max_local
    }])
    return summary, top_cmdb_df, name_counts

# Summaries for each file
metric_summary, metric_top_cmdb, metric_kpi_counts = summarize_df(metric_df, ts_col='timestamp', name_col='kpi_name', name_label='kpi_name', top_cmdb=20, top_names=50)
trace_summary, trace_top_cmdb, trace_name_counts = summarize_df(trace_df, ts_col='timestamp', name_col='trace_name', name_label='trace_name', top_cmdb=20, top_names=50)
log_summary, log_top_cmdb, log_name_counts = summarize_df(log_df, ts_col='timestamp', name_col='log_name', name_label='log_name', top_cmdb=20, top_names=50)
error_summary, error_top_cmdb, error_message_counts = summarize_df(error_logs_df, ts_col='timestamp', name_col='message', name_label='message', top_cmdb=20, top_names=50)

# Keep outputs compact: only show summary rows and top lists (limited as requested)
metric_summary, metric_top_cmdb, metric_kpi_counts, trace_summary, trace_top_cmdb, trace_name_counts, log_summary, log_top_cmdb, log_name_counts, error_summary, error_top_cmdb, error_message_counts

```
Out[1]:
```
summary = (
    "Summary of loaded telemetry (times shown in Asia/Shanghai / UTC+8):\n\n"
    "metric.csv:\n"
    "- Rows: 144,220\n"
    "- Time range: 2022-03-20 10:30:00 to 2022-03-20 10:59:00 (+08:00)\n"
    "- Top cmdb_ids (sample of top entries): adservice (10230), adservice2 (9990), frontend-0 (4800), frontend2-0 (4800), frontend-1 (4680).\n"
    "- KPI name diversity: many distinct kpi_name values (top examples include many container.node-5.* metrics such as container_network_transmit_*, container_network_receive_*, container_cpu_*; a 50-item sample was listed).\n\n"
    "trace.csv:\n"
    "- Rows: 36,216\n"
    "- Time range: 2022-03-20 10:30:00 to 2022-03-20 10:59:00 (+08:00)\n"
    "- Top cmdb_ids (sample): frontend-2 (2760), frontend-0 (2728), frontend-1 (2704), checkoutservice-2 (2688), checkoutservice-1 (2684).\n"
    "- Trace name diversity: many trace.* features; top examples include trace.from_frontend-2.{row_count,error_rate,duration_p95,duration_mean} and many similar per-source/destination trace metrics (50-item sample returned).\n\n"
    "log.csv:\n"
    "- Rows: 1,814\n"
    "- Time range: 2022-03-20 10:30:00 to 2022-03-20 10:59:00 (+08:00)\n"
    "- Top cmdb_ids: many service pods with 60 records each (e.g., adservice-0/1/2, cartservice-0/1/2, checkoutservice-0/1/2, frontend-0/1/2, etc.).\n"
    "- Distinct log names (top): log.error_count (907), log.row_count (907).\n\n"
    "error_logs.csv:\n"
    "- Rows: 1,751\n"
    "- Time range: 2022-03-20 10:30:00 to 2022-03-20 10:59:59 (+08:00)\n"
    "- Top cmdb_ids: frontend-2 (664), frontend-0 (604), frontend-1 (483).\n"
    "- Distinct messages: a single dominant message was present in the sample: \"severity: warning, message: failed to retrieve...\" (count 1,751).\n\n"
    "Notes:\n"
    "- Full DataFrames remain loaded in kernel variables (metric_df, trace_df, log_df, error_logs_df) for follow-up queries.\n"
    "- The above is a compact aggregate summary (top samples). Ask if you want specific expanded lists (e.g., full top-20 cmdb_ids or full top-50 kpi/trace/log names)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     rows   min_epoch   max_epoch min_local (Asia/Shanghai) max_local (Asia/Shanghai)
0  144220  1647743400  1647745140 2022-03-20 10:30:00+08:00 2022-03-20 10:59:00+08:00,                      cmdb_id  count
0                  adservice  10230
1                 adservice2   9990
2                 frontend-0   4800
3                frontend2-0   4800
4                 frontend-1   4680
5                 frontend-2   4380
6          checkoutservice-0   3780
7          checkoutservice-1   3540
8          checkoutservice-2   3540
9    productcatalogservice-0   3480
10   productcatalogservice-2   3480
11   productcatalogservice-1   3360
12        checkoutservice2-0   3300
13  productcatalogservice2-0   2880
14             cartservice-1   2760
15             cartservice-0   2640
16            cartservice2-0   2640
17              adservice2-0   2580
18               adservice-0   2580
19             cartservice-2   2520,                                              kpi_name  count
0   container.node-5.container_network_transmit_MB...   1050
1   container.node-5.container_network_transmit_er...   1050
2   container.node-5.container_network_transmit_pa...   1050
3   container.node-5.container_network_receive_pac...   1050
4   container.node-5.container_network_receive_pac...   1050
..                                                ...    ...
45  container.node-5.container_cpu_cfs_throttled_p...    990
46  container.node-5.container_cpu_cfs_throttled_s...    990
47        container.node-5.container_file_descriptors    990
48         container.node-5.container_memory_usage_MB    990
49  container.node-5.container_fs_io_current./dev/...    990

[50 rows x 2 columns],     rows   min_epoch   max_epoch min_local (Asia/Shanghai) max_local (Asia/Shanghai)
0  36216  1647743400  1647745140 2022-03-20 10:30:00+08:00 2022-03-20 10:59:00+08:00,                     cmdb_id  count
0                frontend-2   2760
1                frontend-0   2728
2                frontend-1   2704
3         checkoutservice-2   2688
4         checkoutservice-1   2684
5         checkoutservice-0   2632
6   productcatalogservice-1   1196
7   productcatalogservice-0   1192
8   productcatalogservice-2   1192
9               frontend2-0   1068
10       checkoutservice2-0    972
11  recommendationservice-0    840
12  recommendationservice-2    840
13  recommendationservice-1    840
14            cartservice-0    840
15            cartservice-2    836
16            cartservice-1    832
17        shippingservice-0    720
18        currencyservice-1    720
19        currencyservice-2    720,                                trace_name  count
0         trace.from_frontend-2.row_count    630
1        trace.from_frontend-2.error_rate    630
2      trace.from_frontend-2.duration_p95    630
3     trace.from_frontend-2.duration_mean    630
4     trace.from_frontend-0.duration_mean    622
..                                    ...    ...
45    trace.from_frontend2-0.duration_p95    207
46      trace.from_frontend2-0.error_rate    207
47       trace.from_frontend2-0.row_count    207
48   trace.to_currencyservice-2.row_count    180
49  trace.to_currencyservice-2.error_rate    180

[50 rows x 2 columns],    rows   min_epoch   max_epoch min_local (Asia/Shanghai) max_local (Asia/Shanghai)
0  1814  1647743400  1647745140 2022-03-20 10:30:00+08:00 2022-03-20 10:59:00+08:00,               cmdb_id  count
0         adservice-0     60
1         adservice-1     60
2         adservice-2     60
3       cartservice-0     60
4       cartservice-1     60
5       cartservice-2     60
6   checkoutservice-0     60
7   checkoutservice-1     60
8   checkoutservice-2     60
9   currencyservice-0     60
10  currencyservice-1     60
11  currencyservice-2     60
12     emailservice-0     60
13     emailservice-1     60
14     emailservice-2     60
15         frontend-0     60
16         frontend-1     60
17         frontend-2     60
18   paymentservice-0     60
19   paymentservice-1     60,           log_name  count
0  log.error_count    907
1    log.row_count    907,    rows   min_epoch   max_epoch min_local (Asia/Shanghai) max_local (Asia/Shanghai)
0  1751  1647743400  1647745199 2022-03-20 10:30:00+08:00 2022-03-20 10:59:59+08:00,       cmdb_id  count
0  frontend-2    664
1  frontend-0    604
2  frontend-1    483,                                              message  count
0  severity: warning, message: failed to retrieve...   1751)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# reuse metric_df already loaded in kernel
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 per (cmdb_id, kpi_name) using ALL rows (store as p95_df)
p95_df = metric_df.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value'].quantile(0.95).reset_index().rename(columns={'value': 'P95_threshold'})

# Count distinct (cmdb_id, kpi_name) pairs in entire metric_df
distinct_pairs_count = p95_df.shape[0]

# 2) Filter metric_df to incident time range (inclusive)
start_ts = 1647743400
end_ts = 1647745140
window_df = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# 3) Merge windowed data with global P95 thresholds (so each row has its group's P95)
merged = window_df.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# create boolean column whether this row is above the global P95
merged['above_p95'] = merged['value'] > merged['P95_threshold']

# 3 continued) aggregate per (cmdb_id, kpi_name) in the window
grouped = merged.groupby(['cmdb_id', 'kpi_name'], dropna=False)
total_points = grouped.size().rename('total_points_in_window')
count_above = grouped['above_p95'].sum().astype(int).rename('count_points_above_P95')
max_value = grouped['value'].max().rename('max_value_in_window')

# earliest timestamp where value > P95 (if any)
earliest_above = merged[merged['above_p95']].groupby(['cmdb_id', 'kpi_name'])['timestamp'].min().rename('earliest_timestamp_above_P95')

# Combine aggregated pieces
agg_df = pd.concat([total_points, count_above, max_value], axis=1).reset_index()
agg_df = agg_df.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')
agg_df = agg_df.merge(earliest_above.reset_index(), on=['cmdb_id', 'kpi_name'], how='left')

# Convert earliest timestamp to Asia/Shanghai timezone-aware datetimes (keep NaT where missing)
agg_df['earliest_timestamp_above_P95'] = pd.to_datetime(agg_df['earliest_timestamp_above_P95'], unit='s', utc=True).dt.tz_convert(tz)

# 4) Order by count_points_above_P95 desc, ties by max_value_in_window desc, select columns and top 20
result_cols = ['cmdb_id', 'kpi_name', 'total_points_in_window', 'count_points_above_P95',
               'earliest_timestamp_above_P95', 'max_value_in_window', 'P95_threshold']
top20_suspicious = agg_df.sort_values(['count_points_above_P95', 'max_value_in_window'], ascending=[False, False])[result_cols].head(20)

# 5) distinct pairs count already computed above
top20_suspicious, distinct_pairs_count

```
Out[2]:
```
Summary (straightforward):

- Total distinct (cmdb_id, kpi_name) pairs in metric_df: 4,818.

- Top suspicious component-KPIs (compact findings from the top-20 results):
  - adservice / adservice2: several JVM/runtime CPU and threading KPIs (e.g., runtime.java_lang_OperatingSystem_ProcessCpuTime, runtime.java_lang_Threading_CurrentThreadCpuTime, runtime.java_lang_Threading_CurrentThreadUserTime, CurrentThreadAllocated…) each has:
    - 30 total points in the incident window
    - 2 points above the globally computed P95
    - earliest above-P95 around 2022-03-20 10:57–10:58 (Asia/Shanghai)
    - max values just slightly above the P95 thresholds (i.e., small exceedances)
  - Nodes (node-1, node-2, node-3, node-4): filesystem and disk KPIs (system.fs.inodes.total, system.fs.inodes.free, system.disk.used) each has:
    - 30 total points in window
    - 2 points above global P95
    - earliest inode-related exceedances at ~2022-03-20 10:34; disk.used exceedances at ~10:58
    - max values also slightly above P95 thresholds

- Interpretation / recommendation:
  - The top candidates are concentrated on adservice/adservice2 runtime CPU/thread metrics (breaches at ~10:57–10:58) and node filesystem/disk metrics (inode breaches earlier at ~10:34, disk usage breaches near ~10:58).
  - All flagged KPIs show very few samples above P95 (only 2 points out of 30), and the max values are only marginally above the P95 thresholds — this suggests short/borderline spikes rather than sustained, large anomalies.
  - Next steps: inspect adservice/adservice2 pods around 10:57–10:58 for transient CPU spikes and check node disk/inode status (possible brief inode pressure or disk usage fluctuation) to confirm impact and root cause.

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id                                           kpi_name  total_points_in_window  count_points_above_P95 earliest_timestamp_above_P95  max_value_in_window  P95_threshold
823   adservice2   runtime.java_lang_OperatingSystem_ProcessCpuTime                      30                       2    2022-03-20 10:58:00+08:00         7.427483e+11   7.424947e+11
244    adservice   runtime.java_lang_OperatingSystem_ProcessCpuTime                      30                       2    2022-03-20 10:58:00+08:00         7.240300e+11   7.237410e+11
832   adservice2   runtime.java_lang_Threading_CurrentThreadCpuTime                      30                       2    2022-03-20 10:58:00+08:00         6.930282e+10   6.927454e+10
253    adservice   runtime.java_lang_Threading_CurrentThreadCpuTime                      30                       2    2022-03-20 10:57:00+08:00         6.844618e+10   6.840278e+10
255    adservice  runtime.java_lang_Threading_CurrentThreadUserTime                      30                       2    2022-03-20 10:57:00+08:00         6.657583e+10   6.652554e+10
834   adservice2  runtime.java_lang_Threading_CurrentThreadUserTime                      30                       2    2022-03-20 10:58:00+08:00         6.638750e+10   6.635971e+10
252    adservice  runtime.java_lang_Threading_CurrentThreadAlloc...                      30                       2    2022-03-20 10:58:00+08:00         5.415224e+10   5.413065e+10
831   adservice2  runtime.java_lang_Threading_CurrentThreadAlloc...                      30                       2    2022-03-20 10:58:00+08:00         5.403666e+10   5.401601e+10
3259      node-4                             system.fs.inodes.total                      30                       2    2022-03-20 10:34:00+08:00         6.796969e+09   6.796280e+09
3257      node-4                              system.fs.inodes.free                      30                       2    2022-03-20 10:34:00+08:00         6.794393e+09   6.793705e+09
3256      node-4                                   system.disk.used                      30                       2    2022-03-20 10:58:00+08:00         5.726636e+09   5.726458e+09
3083      node-1                             system.fs.inodes.total                      30                       2    2022-03-20 10:34:00+08:00         5.107490e+09   5.107007e+09
3142      node-2                             system.fs.inodes.total                      30                       2    2022-03-20 10:34:00+08:00         5.107484e+09   5.107010e+09
3200      node-3                             system.fs.inodes.total                      30                       2    2022-03-20 10:34:00+08:00         5.107321e+09   5.106960e+09
3081      node-1                              system.fs.inodes.free                      30                       2    2022-03-20 10:34:00+08:00         5.105559e+09   5.105076e+09
3140      node-2                              system.fs.inodes.free                      30                       2    2022-03-20 10:34:00+08:00         5.105524e+09   5.105049e+09
3198      node-3                              system.fs.inodes.free                      30                       2    2022-03-20 10:34:00+08:00         5.105382e+09   5.105021e+09
3197      node-3                                   system.disk.used                      30                       2    2022-03-20 10:58:00+08:00         4.304242e+09   4.304123e+09
3080      node-1                                   system.disk.used                      30                       2    2022-03-20 10:58:00+08:00         4.299838e+09   4.299650e+09
3139      node-2                                   system.disk.used                      30                       2    2022-03-20 10:58:00+08:00         4.299396e+09   4.299208e+09, 4818)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse trace_df already loaded in kernel
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P5 per (cmdb_id, trace_name) using ALL rows (store thresholds in trace_thresh_df)
p95 = trace_df.groupby(['cmdb_id', 'trace_name'], dropna=False)['value'].quantile(0.95).rename('P95_threshold')
p5  = trace_df.groupby(['cmdb_id', 'trace_name'], dropna=False)['value'].quantile(0.05).rename('P5_threshold')
trace_thresh_df = pd.concat([p95, p5], axis=1).reset_index()

# Count distinct (cmdb_id, trace_name) pairs
distinct_trace_pairs_count = trace_thresh_df.shape[0]

# 2) Filter trace_df to incident window (inclusive)
start_ts = 1647743400
end_ts   = 1647745140
trace_window = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()

# 3) Merge windowed data with thresholds so each row has P95 and P5
merged = trace_window.merge(trace_thresh_df, on=['cmdb_id', 'trace_name'], how='left')

# Flags for above P95 and below P5
merged['above_p95'] = merged['value'] > merged['P95_threshold']
merged['below_p5']  = merged['value'] < merged['P5_threshold']

# Aggregations in window per (cmdb_id, trace_name)
grp = merged.groupby(['cmdb_id', 'trace_name'], dropna=False)
agg = grp.agg(
    total_points_in_window = ('value', 'size'),
    count_points_above_P95 = ('above_p95', 'sum'),
    count_points_below_P5  = ('below_p5', 'sum'),
    max_value_in_window    = ('value', 'max'),
    min_value_in_window    = ('value', 'min')
).reset_index()

# Earliest timestamps for above-P95 and below-P5
earliest_above = merged[merged['above_p95']].groupby(['cmdb_id','trace_name'])['timestamp'].min().rename('earliest_timestamp_above_P95').reset_index()
earliest_below = merged[merged['below_p5']].groupby(['cmdb_id','trace_name'])['timestamp'].min().rename('earliest_timestamp_below_P5').reset_index()

# Join thresholds and earliest timestamps
agg = agg.merge(trace_thresh_df, on=['cmdb_id','trace_name'], how='left')
agg = agg.merge(earliest_above, on=['cmdb_id','trace_name'], how='left')
agg = agg.merge(earliest_below, on=['cmdb_id','trace_name'], how='left')

# Convert earliest timestamps to Asia/Shanghai timezone-aware datetimes
agg['earliest_timestamp_above_P95'] = pd.to_datetime(agg['earliest_timestamp_above_P95'], unit='s', utc=True).dt.tz_convert(tz)
agg['earliest_timestamp_below_P5']  = pd.to_datetime(agg['earliest_timestamp_below_P5'], unit='s', utc=True).dt.tz_convert(tz)

# 4) Compute selection sort keys:
# primary = max(count_above, count_below)
agg['primary_count'] = agg[['count_points_above_P95', 'count_points_below_P5']].max(axis=1)

# tie-breaker: largest absolute deviation between max/min and corresponding threshold
agg['dev_above'] = (agg['max_value_in_window'] - agg['P95_threshold']).abs()
agg['dev_below'] = (agg['min_value_in_window'] - agg['P5_threshold']).abs()
agg['deviation_score'] = agg[['dev_above','dev_below']].max(axis=1)

# Order and select columns
order_cols = ['primary_count', 'deviation_score']
top20_trace = agg.sort_values(order_cols, ascending=[False, False]).head(20)

result_cols = [
    'cmdb_id', 'trace_name', 'total_points_in_window',
    'count_points_above_P95', 'earliest_timestamp_above_P95',
    'count_points_below_P5', 'earliest_timestamp_below_P5',
    'max_value_in_window', 'min_value_in_window',
    'P95_threshold', 'P5_threshold'
]
top20_trace = top20_trace[result_cols]

# 5) Return compact top20 table and distinct pairs count
top20_trace, distinct_trace_pairs_count

```
Out[3]:
```
Summary (straightforward, times in Asia/Shanghai UTC+8):

- Distinct (cmdb_id, trace_name) pairs in trace_df: 1,232.

- Top suspicious trace findings (based on counts of values above P95 or below P5):
  - Frontend pods (frontend-0, frontend-1, frontend-2, frontend2-0) show anomalies in trace.self.row_count:
    - Each series: 30 points in window, 2 points above the global P95 and 1–2 points below the global P5.
    - Earliest high spikes occurred between ~10:36 and ~10:58; earliest low dips between ~10:31 and ~10:44.
    - Example magnitudes: max row_count up to ~1,157–1,264 with P95 ~1,123–1,233 and P5 ~848–984 (short, borderline deviations).
  - Traces between frontend and productcatalogservice (to_productcatalogservice.* and from_frontend.* for productcatalogserviceX-0/1/2 and productcatalogservice2-0):
    - Each: 30 points, 2 counts above P95 and 1–2 below P5.
    - Max row_counts ~161–198 (P95 ~147–185, P5 ~149–98), earliest above ~10:50–10:51, earliest below ~10:32–10:44.
  - Traces involving currencyservice (trace.from_frontend-2.row_count and trace.to_currencyservice.*):
    - Each: 30 points, 2 above-P95 and 2 below-P5 in several cases.
    - Max values ~143 with P95 ~126–127 and P5 ~89; earliest high ~10:52, earliest low ~10:37.

- Interpretation and recommended next steps:
  - The top items show brief, small-number spikes and dips (typically 2 out of 30 points) rather than sustained large anomalies. Both high and low deviations co-occur across frontend pods and downstream service call counts (productcatalogservice and currencyservice).
  - This pattern suggests transient traffic/throughput fluctuations (short bursts and short drops) impacting frontends and corresponding downstream call volumes around ~10:30–10:58.
  - Recommended checks: inspect frontend pod load and request rates around 10:34–10:58, review routing/load-balancer behavior, and correlate with application logs and pod CPU/network metrics to confirm if these transient trace count deviations caused user impact.

The original code execution output of IPython Kernel is also provided below for reference:

(                       cmdb_id                                   trace_name  total_points_in_window  count_points_above_P95 earliest_timestamp_above_P95  count_points_below_P5 earliest_timestamp_below_P5  max_value_in_window  min_value_in_window  P95_threshold  P5_threshold
667                 frontend-1                         trace.self.row_count                      30                       2    2022-03-20 10:58:00+08:00                      2   2022-03-20 10:44:00+08:00               1043.0                677.0         951.05        692.90
851                frontend2-0                         trace.self.row_count                      30                       2    2022-03-20 10:36:00+08:00                      1   2022-03-20 10:40:00+08:00                619.0                397.0         566.65        451.00
575                 frontend-0                         trace.self.row_count                      30                       2    2022-03-20 10:37:00+08:00                      2   2022-03-20 10:31:00+08:00               1157.0                795.0        1123.85        848.95
759                 frontend-2                         trace.self.row_count                      30                       2    2022-03-20 10:52:00+08:00                      2   2022-03-20 10:37:00+08:00               1264.0                970.0        1233.10        983.80
871                frontend2-0  trace.to_productcatalogservice2-0.row_count                      30                       2    2022-03-20 10:35:00+08:00                      2   2022-03-20 10:32:00+08:00                281.0                178.0         278.10        201.75
1047  productcatalogservice2-0             trace.from_frontend2-0.row_count                      30                       2    2022-03-20 10:35:00+08:00                      2   2022-03-20 10:32:00+08:00                281.0                178.0         278.10        201.75
495          currencyservice-1              trace.from_frontend-2.row_count                      30                       2    2022-03-20 10:52:00+08:00                      2   2022-03-20 10:37:00+08:00                143.0                 87.0         126.65         89.90
519          currencyservice-2              trace.from_frontend-2.row_count                      30                       2    2022-03-20 10:52:00+08:00                      2   2022-03-20 10:37:00+08:00                143.0                 88.0         126.65         89.35
803                 frontend-2         trace.to_currencyservice-1.row_count                      30                       2    2022-03-20 10:52:00+08:00                      2   2022-03-20 10:37:00+08:00                143.0                 87.0         126.65         89.90
807                 frontend-2         trace.to_currencyservice-2.row_count                      30                       2    2022-03-20 10:52:00+08:00                      2   2022-03-20 10:37:00+08:00                143.0                 88.0         126.65         89.35
471          currencyservice-0              trace.from_frontend-2.row_count                      30                       2    2022-03-20 10:52:00+08:00                      2   2022-03-20 10:37:00+08:00                143.0                 88.0         127.10         88.90
799                 frontend-2         trace.to_currencyservice-0.row_count                      30                       2    2022-03-20 10:52:00+08:00                      2   2022-03-20 10:37:00+08:00                143.0                 88.0         127.10         88.90
723                 frontend-1   trace.to_productcatalogservice-1.row_count                      30                       2    2022-03-20 10:51:00+08:00                      2   2022-03-20 10:44:00+08:00                162.0                 93.0         147.10         98.45
979    productcatalogservice-1              trace.from_frontend-1.row_count                      30                       2    2022-03-20 10:51:00+08:00                      2   2022-03-20 10:44:00+08:00                162.0                 93.0         147.10         98.45
727                 frontend-1   trace.to_productcatalogservice-2.row_count                      30                       2    2022-03-20 10:51:00+08:00                      2   2022-03-20 10:44:00+08:00                161.0                 94.0         146.55         98.45
1019   productcatalogservice-2              trace.from_frontend-1.row_count                      30                       2    2022-03-20 10:51:00+08:00                      2   2022-03-20 10:44:00+08:00                161.0                 94.0         146.55         98.45
719                 frontend-1   trace.to_productcatalogservice-0.row_count                      30                       2    2022-03-20 10:51:00+08:00                      2   2022-03-20 10:36:00+08:00                161.0                 93.0         147.55         98.45
939    productcatalogservice-0              trace.from_frontend-1.row_count                      30                       2    2022-03-20 10:51:00+08:00                      2   2022-03-20 10:36:00+08:00                161.0                 93.0         147.55         98.45
811                 frontend-2   trace.to_productcatalogservice-0.row_count                      30                       2    2022-03-20 10:50:00+08:00                      2   2022-03-20 10:34:00+08:00                197.0                148.0         184.65        149.90
819                 frontend-2   trace.to_productcatalogservice-2.row_count                      30                       2    2022-03-20 10:50:00+08:00                      2   2022-03-20 10:34:00+08:00                198.0                147.0         185.65        149.80, 1232)```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# Reuse existing DataFrames from kernel: error_logs_df, log_df
tz = pytz.timezone('Asia/Shanghai')

# Incident window (inclusive)
start_ts = 1647743400
end_ts   = 1647745140

# 1) error_logs_df: filter to window
error_window = error_logs_df[(error_logs_df['timestamp'] >= start_ts) & (error_logs_df['timestamp'] <= end_ts)].copy()
# ensure timestamp numeric
error_window['timestamp'] = pd.to_numeric(error_window['timestamp'], errors='coerce').astype('Int64')

# total entries and earliest timestamp per cmdb_id
err_agg = error_window.groupby('cmdb_id', dropna=False).agg(
    total_error_log_entries_in_window = ('timestamp', 'size'),
    earliest_timestamp_in_window = ('timestamp', 'min')
).reset_index()

# minute with highest count per cmdb_id: compute per-minute counts then pick max per cmdb_id
per_min_counts = error_window.groupby(['cmdb_id', 'timestamp'], dropna=False).size().rename('minute_count').reset_index()
# for ties in minute_count, pick the earliest timestamp (stable)
per_min_counts_sorted = per_min_counts.sort_values(['cmdb_id', 'minute_count', 'timestamp'], ascending=[True, False, True])
max_minute = per_min_counts_sorted.groupby('cmdb_id', as_index=False).first().rename(columns={
    'timestamp': 'timestamp_of_max_minute',
    'minute_count': 'max_minute_count'
})[['cmdb_id', 'timestamp_of_max_minute', 'max_minute_count']]

# merge with err_agg
error_summary = err_agg.merge(max_minute, on='cmdb_id', how='left')

# convert timestamps to Asia/Shanghai timezone-aware datetimes
error_summary['earliest_timestamp_in_window'] = pd.to_datetime(error_summary['earliest_timestamp_in_window'], unit='s', utc=True).dt.tz_convert(tz)
error_summary['timestamp_of_max_minute'] = pd.to_datetime(error_summary['timestamp_of_max_minute'], unit='s', utc=True).dt.tz_convert(tz)

# top 20 cmdb_ids ordered by total_error_log_entries_in_window desc
error_top20 = error_summary.sort_values('total_error_log_entries_in_window', ascending=False).head(20)

# distinct cmdb_id count present in error_logs_df (global, not just window)
error_distinct_cmdb_count = int(error_logs_df['cmdb_id'].nunique())

# 2) log_df: compute global P95 per (cmdb_id, log_name) using ALL rows (store thresholds in kernel)
log_p95_df = log_df.groupby(['cmdb_id', 'log_name'], dropna=False)['value'].quantile(0.95).reset_index().rename(columns={'value':'P95_threshold'})

# Filter log_df to incident window
log_window = log_df[(log_df['timestamp'] >= start_ts) & (log_df['timestamp'] <= end_ts)].copy()
log_window['timestamp'] = pd.to_numeric(log_window['timestamp'], errors='coerce').astype('Int64')

# Merge windowed logs with global P95 thresholds
log_merged = log_window.merge(log_p95_df, on=['cmdb_id', 'log_name'], how='left')

# flag above P95
log_merged['above_p95'] = log_merged['value'] > log_merged['P95_threshold']

# Aggregate per (cmdb_id, log_name) in window
log_grp = log_merged.groupby(['cmdb_id', 'log_name'], dropna=False)
log_agg = log_grp.agg(
    total_points_in_window = ('value', 'size'),
    count_points_above_P95  = ('above_p95', 'sum'),
    max_value_in_window    = ('value', 'max')
).reset_index()

# earliest timestamp above P95 (if any)
earliest_above_log = log_merged[log_merged['above_p95']].groupby(['cmdb_id','log_name'])['timestamp'].min().rename('earliest_timestamp_above_P95').reset_index()

# Merge earliest above and P95
log_agg = log_agg.merge(log_p95_df, on=['cmdb_id','log_name'], how='left')
log_agg = log_agg.merge(earliest_above_log, on=['cmdb_id','log_name'], how='left')

# Convert earliest timestamp to Asia/Shanghai timezone-aware datetimes
log_agg['earliest_timestamp_above_P95'] = pd.to_datetime(log_agg['earliest_timestamp_above_P95'], unit='s', utc=True).dt.tz_convert(tz)

# Order by count_points_above_P95 desc, tie-breaker max_value_in_window desc, take top 20
log_top20 = log_agg.sort_values(['count_points_above_P95', 'max_value_in_window'], ascending=[False, False])[[
    'cmdb_id','log_name','total_points_in_window','count_points_above_P95','earliest_timestamp_above_P95','max_value_in_window','P95_threshold'
]].head(20)

# Return compact results: top20 error summary, top20 log anomalies, and distinct cmdb_id count in error_logs_df
error_top20, log_top20, error_distinct_cmdb_count

```
Out[4]:
```
Summary (plain English, times in Asia/Shanghai UTC+8):

1) error_logs (incident window 2022-03-20 10:30:00 — 10:59:00)
- Top cmdb_ids by error log entries in the window:
  - frontend-2: 640 entries; earliest at 2022-03-20 10:30:00; busiest minute at 2022-03-20 10:34:23 (2 entries).
  - frontend-0: 588 entries; earliest at 2022-03-20 10:30:00; busiest minute at 2022-03-20 10:47:22 (2 entries).
  - frontend-1: 462 entries; earliest at 2022-03-20 10:30:00; busiest minute at 2022-03-20 10:45:16 (2 entries).
- Distinct cmdb_id count in error_logs_df: 3.

2) logs (log.row_count, incident window; using global P95 thresholds)
- Many pods show brief log.row_count spikes above their global P95 (typically 2 points out of 30):
  - frontend-2: total 30 points, 2 above P95, earliest above at 2022-03-20 10:52:00, max=1962 (P95=1940.05).
  - frontend-0: total 30, 2 above, earliest above at 2022-03-20 10:37:00, max=1766 (P95=1725.35).
  - frontend-1: total 30, 2 above, earliest above at 2022-03-20 10:58:00, max=1573 (P95=1474.20).
  - Other affected pods with similar patterns (2/30 points above P95): productcatalogservice-0/1/2, cartservice-0/1/2, currencyservice-0/1/2, recommendationservice-0/1/2, adservice-0/1/2, shippingservice-1/2, etc. (top-20 shown in the table).
- These are short/borderline spikes (2 samples exceed P95; max values are only modestly above P95).

Interpretation and recommended next steps:
- Frontend pods are the dominant source of error log entries during the incident window and show transient increases in log.row_count across many downstream services.
- The pattern indicates brief traffic/throughput spikes rather than sustained failures.
- Next steps: inspect frontend request rates and application logs around the noted spike times (10:34–10:58), check load-balancer/routing behavior, and correlate with CPU/network/trace anomalies to confirm root cause.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id  total_error_log_entries_in_window earliest_timestamp_in_window   timestamp_of_max_minute  max_minute_count
2  frontend-2                                640    2022-03-20 10:30:00+08:00 2022-03-20 10:34:23+08:00                 2
0  frontend-0                                588    2022-03-20 10:30:00+08:00 2022-03-20 10:47:22+08:00                 2
1  frontend-1                                462    2022-03-20 10:30:00+08:00 2022-03-20 10:45:16+08:00                 2,                     cmdb_id       log_name  total_points_in_window  count_points_above_P95 earliest_timestamp_above_P95  max_value_in_window  P95_threshold
35               frontend-2  log.row_count                      30                       2    2022-03-20 10:52:00+08:00               1962.0        1940.05
31               frontend-0  log.row_count                      30                       2    2022-03-20 10:37:00+08:00               1766.0        1725.35
33               frontend-1  log.row_count                      30                       2    2022-03-20 10:58:00+08:00               1573.0        1474.20
1               adservice-0  log.row_count                      30                       2    2022-03-20 10:55:00+08:00               1492.0        1416.10
43  productcatalogservice-0  log.row_count                      30                       2    2022-03-20 10:57:00+08:00               1219.0        1159.15
47  productcatalogservice-2  log.row_count                      30                       2    2022-03-20 10:57:00+08:00               1204.0        1166.80
45  productcatalogservice-1  log.row_count                      30                       2    2022-03-20 10:57:00+08:00               1185.0        1132.60
7             cartservice-0  log.row_count                      30                       2    2022-03-20 10:57:00+08:00               1060.0        1015.90
11            cartservice-2  log.row_count                      30                       2    2022-03-20 10:57:00+08:00               1043.0        1011.30
9             cartservice-1  log.row_count                      30                       2    2022-03-20 10:57:00+08:00               1042.0        1010.55
19        currencyservice-0  log.row_count                      30                       2    2022-03-20 10:50:00+08:00                987.0         951.65
23        currencyservice-2  log.row_count                      30                       2    2022-03-20 10:50:00+08:00                987.0         958.95
21        currencyservice-1  log.row_count                      30                       2    2022-03-20 10:50:00+08:00                983.0         952.05
53  recommendationservice-2  log.row_count                      30                       2    2022-03-20 10:57:00+08:00                259.0         245.95
49  recommendationservice-0  log.row_count                      30                       2    2022-03-20 10:57:00+08:00                255.0         246.95
51  recommendationservice-1  log.row_count                      30                       2    2022-03-20 10:57:00+08:00                255.0         242.55
5               adservice-2  log.row_count                      30                       2    2022-03-20 10:55:00+08:00                140.0         133.10
3               adservice-1  log.row_count                      30                       2    2022-03-20 10:55:00+08:00                136.0         129.10
61        shippingservice-2  log.row_count                      30                       2    2022-03-20 10:48:00+08:00                132.0         127.20
59        shippingservice-1  log.row_count                      30                       2    2022-03-20 10:50:00+08:00                126.0         123.20, 3)```
```

In [ ]:
# In[5]:

import pandas as pd
import re
import pytz

# Reuse metric_df and p95_df from kernel
tz = pytz.timezone('Asia/Shanghai')

# Incident window (reuse if defined; otherwise set)
start_ts = 1647743400
end_ts   = 1647745140

# Helper to extract node-id substring like 'node-3' from kpi_name
node_pattern = r'(node-\d+)'

# 1) Frontend pods mapping
frontend_pods = ['frontend-0', 'frontend-1', 'frontend-2', 'frontend2-0']
# Filter metric rows for these pods
frontend_rows = metric_df[metric_df['cmdb_id'].isin(frontend_pods)].copy()
# extract node id occurrences from kpi_name
frontend_rows['node_id'] = frontend_rows['kpi_name'].str.extract(node_pattern, expand=False)
# Keep only rows where a node was found
frontend_node_map = (
    frontend_rows[~frontend_rows['node_id'].isna()]
    .groupby(['cmdb_id', 'node_id'], dropna=False)
    .size()
    .reset_index(name='count_of_metrics_on_that_node')
    .sort_values(['cmdb_id', 'count_of_metrics_on_that_node'], ascending=[True, False])
)
# Limit to top 50 rows for compactness
frontend_node_map = frontend_node_map.head(50)

# 2) Adservice pods mapping (include adservice2 variants if present)
adservice_candidates = ['adservice-0', 'adservice-1', 'adservice-2', 'adservice2-0', 'adservice2']
# intersect with actual cmdb_ids present to avoid nonexistent ids
present_adservices = [c for c in adservice_candidates if c in metric_df['cmdb_id'].unique()]
ad_rows = metric_df[metric_df['cmdb_id'].isin(present_adservices)].copy()
ad_rows['node_id'] = ad_rows['kpi_name'].str.extract(node_pattern, expand=False)
adservice_node_map = (
    ad_rows[~ad_rows['node_id'].isna()]
    .groupby(['cmdb_id', 'node_id'], dropna=False)
    .size()
    .reset_index(name='count_of_metrics_on_that_node')
    .sort_values(['cmdb_id', 'count_of_metrics_on_that_node'], ascending=[True, False])
)
adservice_node_map = adservice_node_map.head(50)

# 3) For each unique node_id discovered above, report node-level KPIs of interest
# gather node ids from both maps
nodes_found = pd.concat([frontend_node_map['node_id'], adservice_node_map['node_id']]).dropna().unique().tolist()

# Normalize node ids (ensure strings like 'node-1')
nodes_found = [n for n in nodes_found if isinstance(n, str)]

# Node-level KPIs of interest
node_kpis = ['system.fs.inodes.total', 'system.fs.inodes.free', 'system.disk.used']

# Filter metric_df to incident window and nodes of interest
metric_window = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# Keep only rows where cmdb_id is one of nodes_found and kpi_name is in node_kpis
node_metrics_window = metric_window[(metric_window['cmdb_id'].isin(nodes_found)) & (metric_window['kpi_name'].isin(node_kpis))].copy()

# Merge with global P95 thresholds (p95_df) which contains P95 per (cmdb_id,kpi_name)
# Ensure p95_df exists in kernel (it was computed earlier)
merged_node = node_metrics_window.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# Flag above P95
merged_node['above_p95'] = merged_node['value'] > merged_node['P95_threshold']

# Aggregate per (node, kpi)
grp = merged_node.groupby(['cmdb_id', 'kpi_name'], dropna=False)
node_kpi_agg = grp.agg(
    total_points_in_window = ('value', 'size'),
    count_points_above_P95  = ('above_p95', 'sum'),
    max_value_in_window    = ('value', 'max')
).reset_index()

# earliest timestamp above P95 (if any)
earliest_above = merged_node[merged_node['above_p95']].groupby(['cmdb_id', 'kpi_name'])['timestamp'].min().rename('earliest_timestamp_above_P95').reset_index()
node_kpi_agg = node_kpi_agg.merge(earliest_above, on=['cmdb_id','kpi_name'], how='left')

# Convert earliest timestamp to Asia/Shanghai timezone-aware datetimes
node_kpi_agg['earliest_timestamp_above_P95'] = pd.to_datetime(node_kpi_agg['earliest_timestamp_above_P95'], unit='s', utc=True).dt.tz_convert(tz)

# Merge P95 threshold for clarity
node_kpi_agg = node_kpi_agg.merge(p95_df, on=['cmdb_id','kpi_name'], how='left')

# Order by count_points_above_P95 desc and limit top 20
node_kpi_summary = node_kpi_agg.sort_values(['count_points_above_P95', 'max_value_in_window'], ascending=[False, False])[
    ['cmdb_id','kpi_name','total_points_in_window','count_points_above_P95','earliest_timestamp_above_P95','max_value_in_window','P95_threshold']
].head(20)

# Display compact results: frontend_node_map, adservice_node_map, node_kpi_summary
frontend_node_map, adservice_node_map, node_kpi_summary

```
Out[5]:
```
Short summary (plain English, Asia/Shanghai times):

1) Where pods reference nodes (counts of metric rows referencing node-* in kpi_name)
- Frontend pods:
  - frontend-0: references node-5 (1920 rows)
  - frontend-1: references node-5 (1920 rows)
  - frontend-2: references node-5 (1920 rows)
  - frontend2-0: references node-6 (1920 rows)
- Adservice pods:
  - adservice-0: references node-5 (1920 rows)
  - adservice-1: references node-5 (1920 rows)
  - adservice-2: references node-5 (1800 rows)
  - adservice2-0: references node-6 (1920 rows)

Conclusion: Most frontend and adservice pods run on node-5. Exceptions: frontend2-0 and adservice2-0 run on node-6.

2) Node-level KPIs of interest (incident window 2022-03-20 10:30:00 – 10:59:00)
- node-6:
  - system.fs.inodes.total: 30 points, 2 points > global P95, earliest above-P95 at 2022-03-20 10:34:00, max ≈ 2.935282e9 (P95 ≈ 2.935135e9).
  - system.fs.inodes.free: 30 points, 2 points > global P95, earliest above-P95 at 2022-03-20 10:34:00, max ≈ 2.933887e9 (P95 ≈ 2.933741e9).
  - system.disk.used: 30 points, 2 points > global P95, earliest above-P95 at 2022-03-20 10:58:00, max ≈ 2.526005e9 (P95 ≈ 2.525868e9).
- node-5:
  - system.disk.used: 30 points, 2 points > global P95, earliest above-P95 at 2022-03-20 10:36:00, max ≈ 4.435968e7 (P95 ≈ 4.435102e7).
  - system.fs.inodes.total: 30 points, 0 points > P95 (max equals P95 in the window).
  - system.fs.inodes.free: 30 points, 0 points > P95 (max equals P95 in the window).

3) Timing alignment and interpretation
- Early (~10:34): node-6 shows inode-related exceedances (system.fs.inodes.total and .free) at ~10:34. The pods on node-6 are frontend2-0 and adservice2-0 — these two pods are the ones most likely impacted by the early inode anomaly.
- Mid (~10:36): node-5 shows a brief disk.used exceedance at ~10:36. Since most frontends and adservice pods run on node-5, that spike could briefly affect the bulk of frontend/adservice traffic.
- Late (~10:58): node-6 shows a disk.used exceedance at ~10:58 (also a late spike). This is a separate late event on node-6.

Overall conclusion and recommended next steps
- Node-6 experienced early inode pressure (~10:34) and a later disk usage spike (~10:58). Only frontend2-0 and adservice2-0 run on node-6, so investigate those pods first for early impacts.
- Node-5, hosting most frontends/adservices, had a disk.used spike around ~10:36 but no inode exceedance in the window; check whether that mid-window disk spike caused transient latency or errors observed across frontends.
- Next steps: examine pod logs and resource metrics for frontend2-0 and adservice2-0 around 10:34 (inodes) and 10:58 (disk), and check I/O/disk metrics and scheduling/eviction events on node-5 around 10:36 to confirm user impact.

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id node_id  count_of_metrics_on_that_node
0   frontend-0  node-5                           1920
1   frontend-1  node-5                           1920
2   frontend-2  node-5                           1920
3  frontend2-0  node-6                           1920,         cmdb_id node_id  count_of_metrics_on_that_node
0   adservice-0  node-5                           1920
1   adservice-1  node-5                           1920
2   adservice-2  node-5                           1800
3  adservice2-0  node-6                           1920,   cmdb_id                kpi_name  total_points_in_window  count_points_above_P95 earliest_timestamp_above_P95  max_value_in_window  P95_threshold
5  node-6  system.fs.inodes.total                      30                       2    2022-03-20 10:34:00+08:00         2.935282e+09   2.935135e+09
4  node-6   system.fs.inodes.free                      30                       2    2022-03-20 10:34:00+08:00         2.933887e+09   2.933741e+09
3  node-6        system.disk.used                      30                       2    2022-03-20 10:58:00+08:00         2.526005e+09   2.525868e+09
0  node-5        system.disk.used                      30                       2    2022-03-20 10:36:00+08:00         4.435968e+07   4.435102e+07
2  node-5  system.fs.inodes.total                      30                       0                          NaT         3.932160e+07   3.932160e+07
1  node-5   system.fs.inodes.free                      30                       0                          NaT         3.899858e+07   3.899858e+07)```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# Reuse kernel DataFrames: error_logs_df, log_df, and log_p95_df if present
tz = pytz.timezone('Asia/Shanghai')

# Incident window
start_ts = 1647743400
end_ts   = 1647745140

# Targets
targets = ["frontend2-0", "adservice2-0"]
log_names_of_interest = ["log.row_count", "log.error_count"]

# 1) error_logs_df: filter to window and targets
error_window = error_logs_df[(error_logs_df['timestamp'] >= start_ts) & (error_logs_df['timestamp'] <= end_ts)].copy()
err_targets = error_window[error_window['cmdb_id'].isin(targets)].copy()

# Aggregates: total entries and earliest timestamp per cmdb_id
err_agg = err_targets.groupby('cmdb_id', dropna=False).agg(
    total_error_log_entries_in_window = ('timestamp', 'size'),
    earliest_timestamp_in_window = ('timestamp', 'min')
).reset_index()

# Top up to 5 distinct messages with counts per cmdb_id
msg_counts = err_targets.groupby(['cmdb_id', 'message'], dropna=False).size().rename('count').reset_index()
msg_top5 = msg_counts.sort_values(['cmdb_id', 'count'], ascending=[True, False]).groupby('cmdb_id').head(5)

# create compact message summary string "msg_short (count); ..."
def make_msg_summary(df):
    pieces = []
    for _, row in df.iterrows():
        msg = str(row['message']).replace('\n',' ').strip()
        if len(msg) > 120:
            msg = msg[:117] + '...'
        pieces.append(f"{msg} ({int(row['count'])})")
    return "; ".join(pieces)

# apply grouping to build summary series, then convert to DataFrame
if not msg_top5.empty:
    msg_summary_series = msg_top5.groupby('cmdb_id').apply(make_msg_summary)
    msg_summary = msg_summary_series.reset_index()
    msg_summary.columns = ['cmdb_id', 'top_messages']
else:
    msg_summary = pd.DataFrame(columns=['cmdb_id','top_messages'])

# Combine error summary
error_summary_targets = err_agg.merge(msg_summary, on='cmdb_id', how='left')

# Ensure rows for both targets exist (even if zero)
for t in targets:
    if t not in error_summary_targets['cmdb_id'].values:
        error_summary_targets = pd.concat([error_summary_targets, pd.DataFrame([{
            'cmdb_id': t,
            'total_error_log_entries_in_window': 0,
            'earliest_timestamp_in_window': pd.NA,
            'top_messages': ''
        }])], ignore_index=True)

# TZ convert earliest timestamp
error_summary_targets['earliest_timestamp_in_window'] = pd.to_datetime(error_summary_targets['earliest_timestamp_in_window'], unit='s', utc=True).dt.tz_convert(tz)
error_summary_targets = error_summary_targets[['cmdb_id','total_error_log_entries_in_window','earliest_timestamp_in_window','top_messages']].sort_values('cmdb_id').head(20)

# 2) log_df: ensure global P95 thresholds exist in kernel; compute if missing
try:
    log_p95_df
except NameError:
    log_p95_df = log_df.groupby(['cmdb_id', 'log_name'], dropna=False)['value'].quantile(0.95).reset_index().rename(columns={'value':'P95_threshold'})

# Filter log_df to window and targets/log_names
log_window = log_df[(log_df['timestamp'] >= start_ts) & (log_df['timestamp'] <= end_ts)].copy()
log_window_targets = log_window[log_window['cmdb_id'].isin(targets) & log_window['log_name'].isin(log_names_of_interest)].copy()

# Merge with global P95 thresholds
log_merged = log_window_targets.merge(log_p95_df, on=['cmdb_id','log_name'], how='left')

# Flag above P95
log_merged['above_p95'] = log_merged['value'] > log_merged['P95_threshold']

# Aggregate per (cmdb_id, log_name)
log_grp = log_merged.groupby(['cmdb_id','log_name'], dropna=False)
log_agg = log_grp.agg(
    total_points_in_window = ('value','size'),
    count_points_above_P95  = ('above_p95','sum'),
    max_value_in_window    = ('value','max')
).reset_index()

# earliest timestamp above P95
earliest_above_log = log_merged[log_merged['above_p95']].groupby(['cmdb_id','log_name'])['timestamp'].min().rename('earliest_timestamp_above_P95').reset_index()
log_agg = log_agg.merge(earliest_above_log, on=['cmdb_id','log_name'], how='left')

# Merge P95_threshold for clarity
log_agg = log_agg.merge(log_p95_df, on=['cmdb_id','log_name'], how='left')

# TZ convert earliest timestamps
log_agg['earliest_timestamp_above_P95'] = pd.to_datetime(log_agg['earliest_timestamp_above_P95'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure rows exist for all combinations of targets x log_names (fill zeros if missing)
all_combos = pd.MultiIndex.from_product([targets, log_names_of_interest], names=['cmdb_id','log_name']).to_frame(index=False)
log_summary_targets = all_combos.merge(log_agg, on=['cmdb_id','log_name'], how='left').fillna({
    'total_points_in_window': 0,
    'count_points_above_P95': 0,
    'max_value_in_window': 0,
    'P95_threshold': pd.NA,
    'earliest_timestamp_above_P95': pd.NaT
})

# cast counts to int
log_summary_targets['total_points_in_window'] = log_summary_targets['total_points_in_window'].astype(int)
log_summary_targets['count_points_above_P95'] = log_summary_targets['count_points_above_P95'].astype(int)

log_summary_targets = log_summary_targets[['cmdb_id','log_name','total_points_in_window','count_points_above_P95','earliest_timestamp_above_P95','max_value_in_window','P95_threshold']].sort_values(['cmdb_id','log_name']).head(20)

# Return compact results
error_summary_targets, log_summary_targets

```
Out[6]:
```
summary = (
    "Findings for frontend2-0 and adservice2-0 in the incident window (2022-03-20 10:30:00 — 10:59:00 +08:00):\n\n"
    "- error_logs: both frontend2-0 and adservice2-0 have 0 error log entries in the window.\n"
    "- logs: for both pods, both log.row_count and log.error_count show 0 points in the window (no samples > P95 nor any max values recorded in these tables).\n\n"
    "Conclusion:\n"
    "- There are no observable log or error entries for frontend2-0 and adservice2-0 in the provided error_logs_df and log_df during the incident window. Therefore the early inode anomaly on node-6 (seen in metrics) did not produce detectable log/error symptoms in these two pods in the available log datasets.\n\n"
    "Possible reasons & next steps:\n"
    "- The anomaly may not have impacted application logging (transient or lower-level system issue).\n"
    "- Logs from these pods might not have been collected or could be outside the analyzed time window—check raw pod logs and collection pipeline.\n"
    "- Correlate with pod metrics/traces (CPU, I/O, traces) and node events/kubelet logs around ~10:34 and ~10:58 to confirm impact and root cause."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(        cmdb_id  total_error_log_entries_in_window earliest_timestamp_in_window top_messages
1  adservice2-0                                  0                          NaT             
0   frontend2-0                                  0                          NaT             ,         cmdb_id         log_name  total_points_in_window  count_points_above_P95 earliest_timestamp_above_P95  max_value_in_window  P95_threshold
3  adservice2-0  log.error_count                       0                       0                          NaT                  0.0            NaN
2  adservice2-0    log.row_count                       0                       0                          NaT                  0.0            NaN
1   frontend2-0  log.error_count                       0                       0                          NaT                  0.0            NaN
0   frontend2-0    log.row_count                       0                       0                          NaT                  0.0            NaN)```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

# Reuse trace_df and trace_thresh_df from kernel if available
tz = pytz.timezone('Asia/Shanghai')

# Ensure global thresholds exist (compute if missing)
try:
    trace_thresh_df
except NameError:
    p95 = trace_df.groupby(['cmdb_id', 'trace_name'], dropna=False)['value'].quantile(0.95).rename('P95_threshold')
    p5  = trace_df.groupby(['cmdb_id', 'trace_name'], dropna=False)['value'].quantile(0.05).rename('P5_threshold')
    trace_thresh_df = pd.concat([p95, p5], axis=1).reset_index()

# Incident window
start_ts = 1647743400
end_ts   = 1647745140

# Target cmdb_ids and trace_name substrings
cmdb_targets = [
    "frontend-0","frontend-1","frontend-2","frontend2-0",
    "adservice-0","adservice-1","adservice-2","adservice2-0",
    "productcatalogservice-0","productcatalogservice-1","productcatalogservice-2","productcatalogservice2-0",
    "currencyservice-0","currencyservice-1","currencyservice-2"
]
substrs = ['error_rate','duration_mean','duration_p95']

# 1) Filter trace_df to window and targets & trace_name substrings
trace_window = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()
mask_cmdb = trace_window['cmdb_id'].isin(cmdb_targets)
mask_name = trace_window['trace_name'].str.contains('|'.join(substrs), na=False)
trace_sel = trace_window[mask_cmdb & mask_name].copy()

# 2) Merge with global P95 thresholds (ensuring P95 exists)
trace_sel = trace_sel.merge(trace_thresh_df[['cmdb_id','trace_name','P95_threshold']], on=['cmdb_id','trace_name'], how='left')

# Flag above P95
trace_sel['above_p95'] = trace_sel['value'] > trace_sel['P95_threshold']

# 3) Aggregate per (cmdb_id, trace_name)
grp = trace_sel.groupby(['cmdb_id','trace_name'], dropna=False)
agg = grp.agg(
    total_points_in_window = ('value','size'),
    count_points_above_P95  = ('above_p95','sum'),
    max_value_in_window    = ('value','max')
).reset_index()

# earliest timestamp above P95 if any
earliest_above = trace_sel[trace_sel['above_p95']].groupby(['cmdb_id','trace_name'])['timestamp'].min().rename('earliest_timestamp_above_P95').reset_index()
agg = agg.merge(earliest_above, on=['cmdb_id','trace_name'], how='left')

# Merge P95_threshold for display
agg = agg.merge(trace_thresh_df[['cmdb_id','trace_name','P95_threshold']], on=['cmdb_id','trace_name'], how='left')

# Convert earliest timestamp to Asia/Shanghai timezone-aware datetimes
agg['earliest_timestamp_above_P95'] = pd.to_datetime(agg['earliest_timestamp_above_P95'], unit='s', utc=True).dt.tz_convert(tz)

# 4) Order and select top 20 rows
top20_trace_subset = agg.sort_values(['count_points_above_P95','max_value_in_window'], ascending=[False, False])[
    ['cmdb_id','trace_name','total_points_in_window','count_points_above_P95','earliest_timestamp_above_P95','max_value_in_window','P95_threshold']
].head(20)

# 5) Number of (cmdb_id, trace_name) pairs examined
examined_pairs_count = agg.shape[0]

# Return compact results
top20_trace_subset, examined_pairs_count

```
Out[7]:
```
Summary (plain English, Asia/Shanghai / UTC+8):

- Total (cmdb_id, trace_name) pairs examined: 414.

- Main findings (top suspicious traces):
  - Numerous frontend → checkoutservice traces show brief latency spikes (both duration_mean and duration_p95). Examples:
    - frontend-1.trace.to_checkoutservice-0.duration_mean & .duration_p95: 23 points in window, 2 points above global P95, earliest high at 10:31.
    - frontend-1.trace.to_checkoutservice-1.*: 26 points, 2 above P95, earliest high at 10:34.
    - frontend-0 and frontend-2 also show multiple trace.to_checkoutservice.* duration_mean/p95 series with 2 points above P95 at times around 10:30–10:37 and later (~10:49–10:52).
  - trace.from_root.duration_p95 for frontends (frontend-0/1/2 and frontend2-0) also shows 2 points above P95 in several series (earliest highs around 10:34–10:41).
  - Magnitudes are small absolute durations (roughly 0.07–0.12 s), and the exceedances are few (typically 2 points per series out of ~23–30).

- Timing pattern vs node issues:
  - High-count trace spikes appear in two clusters: early (~10:30–10:37, e.g., 10:31, 10:34) and later (~10:49–10:52).
  - This aligns roughly with earlier metric findings: node-6 inode anomalies at ~10:34, node-5 disk spike ~10:36, and node-6 disk spike near ~10:58 — suggesting the trace spikes could be temporally correlated with the node-level events.

- Interpretation and recommended next steps:
  - The trace anomalies are transient and limited (2/30 samples), indicating short-lived latency increases rather than sustained outages.
  - They are concentrated on frontend calls to checkoutservice (and root-level request durations on frontends), matching the system areas that showed node inode/disk events.
  - Recommend correlating these specific timestamps (10:30–10:37 and 10:49–10:58) with:
    - node disk/inode events and kubelet/node logs,
    - frontend and checkoutservice pod logs,
    - pod CPU/I/O metrics and network metrics,
    to confirm causality and scope of user impact.

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id                                trace_name  total_points_in_window  count_points_above_P95 earliest_timestamp_above_P95  max_value_in_window  P95_threshold
177   frontend-1  trace.to_checkoutservice-0.duration_mean                      23                       2    2022-03-20 10:31:00+08:00             0.101199       0.047127
178   frontend-1   trace.to_checkoutservice-0.duration_p95                      23                       2    2022-03-20 10:31:00+08:00             0.101199       0.055318
180   frontend-1  trace.to_checkoutservice-1.duration_mean                      26                       2    2022-03-20 10:34:00+08:00             0.099661       0.054843
181   frontend-1   trace.to_checkoutservice-1.duration_p95                      26                       2    2022-03-20 10:34:00+08:00             0.099661       0.069765
112   frontend-0   trace.to_checkoutservice-1.duration_p95                      28                       2    2022-03-20 10:30:00+08:00             0.097191       0.076444
247   frontend-2   trace.to_checkoutservice-0.duration_p95                      30                       2    2022-03-20 10:52:00+08:00             0.096269       0.085282
115   frontend-0   trace.to_checkoutservice-2.duration_p95                      27                       2    2022-03-20 10:32:00+08:00             0.095497       0.090989
108   frontend-0  trace.to_checkoutservice-0.duration_mean                      27                       2    2022-03-20 10:34:00+08:00             0.091417       0.065549
109   frontend-0   trace.to_checkoutservice-0.duration_p95                      27                       2    2022-03-20 10:34:00+08:00             0.091417       0.080262
252   frontend-2  trace.to_checkoutservice-2.duration_mean                      30                       2    2022-03-20 10:49:00+08:00             0.090243       0.056137
253   frontend-2   trace.to_checkoutservice-2.duration_p95                      30                       2    2022-03-20 10:49:00+08:00             0.090243       0.073600
246   frontend-2  trace.to_checkoutservice-0.duration_mean                      30                       2    2022-03-20 10:37:00+08:00             0.086935       0.076309
114   frontend-0  trace.to_checkoutservice-2.duration_mean                      27                       2    2022-03-20 10:32:00+08:00             0.084813       0.068008
250   frontend-2   trace.to_checkoutservice-1.duration_p95                      30                       2    2022-03-20 10:35:00+08:00             0.082593       0.060232
154   frontend-1              trace.from_root.duration_p95                      30                       2    2022-03-20 10:38:00+08:00             0.074366       0.070556
292  frontend2-0              trace.from_root.duration_p95                      30                       2    2022-03-20 10:34:00+08:00             0.074137       0.072576
85    frontend-0              trace.from_root.duration_p95                      30                       2    2022-03-20 10:40:00+08:00             0.071404       0.069951
304  frontend2-0  trace.to_checkoutservice2-0.duration_p95                      27                       2    2022-03-20 10:34:00+08:00             0.071085       0.054265
223   frontend-2              trace.from_root.duration_p95                      30                       2    2022-03-20 10:41:00+08:00             0.070233       0.065953
111   frontend-0  trace.to_checkoutservice-1.duration_mean                      28                       2    2022-03-20 10:30:00+08:00             0.069391       0.057985, 414)```
```